# Day 8：亲眼看懂RMSNorm与RoPE

先读 [Day 8图解](../docs/DAY08_RMSNORM_ROPE.md) 和 [学习记录](../docs/DAY08_STUDY_LOG.md)。
RMSNorm像调节每一行信号音量；RoPE像按位置转动Q/K特征对。
CPU FP64公式oracle与FP32/低精度参考验证，不加载Qwen3权重，不测GPU速度。
Mini RMSNorm内部明确使用FP32，即使.double()也不升级内部归约精度；容差按算法精度设定。
先写预测，另建自己的实现，再与参考对齐；运行本Notebook不等于已独立实现。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/mini_transformer/model.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from learning.mini_transformer.model import RMSNorm, rope
from reasoning_from_scratch.qwen3 import RMSNorm as RepositoryRMSNorm
from reasoning_from_scratch.qwen3 import apply_rope, compute_rope_params

## 1. RMSNorm：先手算，再对齐

输入`[3,4]`，先预测`mean(x**2)`、RMS和不带gamma的输出。
RMSNorm不减均值；`gamma`这里只用全1。
RMSNorm=Root Mean Square Layer Normalization，RoPE=Rotary Position Embedding。
先用FP64公式手算，再与Mini的FP32内部计算作合理容差比较，不将FP64输入误当高精度实现。

In [ ]:
x = torch.tensor([[3., 4.]], dtype=torch.float64)
eps = 1e-6
manual_rms = torch.sqrt(x.square().mean(-1, keepdim=True) + eps)
manual = x / manual_rms
print('mean(x^2):', x.square().mean().item())
print('RMS:', manual_rms.item())
print('RMSNorm(x):', manual.tolist())
assert manual.shape == x.shape
assert manual.mean().item() > 0
norm = RMSNorm(2)
actual = norm(x.float()).detach().double()
torch.testing.assert_close(actual, manual, atol=1e-6, rtol=1e-5)
print('Mini FP32最大误差:', (actual - manual).abs().max().item())

### 边界输入：不要只测随机正态

全零依靠eps保持有限；常数向量不变成零中心；大/小值仍应有限。
独立PyTorch公式作为oracle，不只比较字符串或打印结果。

In [ ]:
boundary = torch.tensor([[0., 0., 0.], [2., 2., 2.], [1e4, -1e4, 1e4], [1e-4, -2e-4, 3e-4]], dtype=torch.float64)
norm3 = RMSNorm(3)
expected = boundary / torch.sqrt(boundary.square().mean(-1, keepdim=True) + eps)
got = norm3(boundary.float()).detach().double()
torch.testing.assert_close(got, expected, atol=1e-6, rtol=1e-5)
assert torch.isfinite(got).all()
print('input:', boundary.tolist())
print('output:', got.tolist())
print('constant row mean:', got[1].mean().item(), '(RMSNorm不是零中心LayerNorm)')
print('边界最大误差:', (got - expected).abs().max().item())

### 非全1的gamma、低精度和梯度

固定eps=1e-6、gamma=[0.5,1,1.5]，分别测试FP32/FP16/BF16，oracle由输入的实际量化值转FP64计算。
对照Mini、仓库RMSNorm与torch.nn.RMSNorm；不使用后者的默认eps。
gamma是逐特征增益，输出不要求零均值或严格单位RMS。CPU测试不证明GPU性能。

In [ ]:
gamma = torch.tensor([0.5, 1., 1.5])
for dtype, atol, rtol in ((torch.float32, 1e-6, 1e-5),
                         (torch.float16, 2e-3, 2e-3),
                         (torch.bfloat16, 2e-2, 1e-2)):
    sample = boundary.to(dtype)
    oracle_input = sample.double()
    oracle = oracle_input / torch.sqrt(oracle_input.square().mean(-1, keepdim=True) + eps)
    oracle = oracle * gamma.double()
    mini = RMSNorm(3, eps=eps)
    repo = RepositoryRMSNorm(3, eps=eps)
    builtin = torch.nn.RMSNorm(3, eps=eps, dtype=dtype)
    with torch.no_grad():
        mini.weight.copy_(gamma)
        repo.scale.copy_(gamma)
        builtin.weight.copy_(gamma)
    for name, implementation in (('mini', mini), ('repository', repo), ('torch', builtin)):
        output = implementation(sample)
        assert output.dtype == dtype and torch.isfinite(output).all()
        torch.testing.assert_close(output.double(), oracle, atol=atol, rtol=rtol)
        print(dtype, name, 'max_error=', (output.double() - oracle).abs().max().item())
train_x = torch.tensor([[[1., 2., 3.], [-2., 1., 4.]]], requires_grad=True)
train_norm = RMSNorm(3)
train_norm(train_x).square().mean().backward()
assert torch.isfinite(train_x.grad).all() and torch.isfinite(train_norm.weight.grad).all()
print('输入/gamma梯度存在且有限：passed')

## 2. RoPE：位置改变角度，shape不变

创建`[B,H,T,D]=[1,2,3,4]`，比较offset=0和offset=5。
先预测：两者shape是否相同？数值是否相同？D为4意味着每个head有2个旋转对。
本仓库采用split-halves：配对(0,2)、(1,3)，不是相邻配对。theta明确统一为10000。

In [ ]:
generator = torch.Generator().manual_seed(8)
q = torch.randn(1, 2, 3, 4, dtype=torch.float32, generator=generator)
theta = 10_000.0
rot0 = rope(q, start=0, theta=theta)
rot5 = rope(q, start=5, theta=theta)
assert rot0.shape == rot5.shape == q.shape
assert not torch.allclose(rot0, rot5)
cos, sin = compute_rope_params(4, theta_base=theta, context_length=16)
ref0 = apply_rope(q, cos, sin, 0)
ref5 = apply_rope(q, cos, sin, 5)
torch.testing.assert_close(rot0, ref0, atol=1e-6, rtol=1e-6)
torch.testing.assert_close(rot5, ref5, atol=1e-6, rtol=1e-6)
print('input shape:', tuple(q.shape))
print('offset0/5 shape:', tuple(rot0.shape), tuple(rot5.shape))
print('max difference:', (rot0-rot5).abs().max().item())
torch.testing.assert_close(rot5.norm(dim=-1), q.norm(dim=-1), atol=1e-6, rtol=1e-5)
torch.testing.assert_close(rot0[:, :, :1], q[:, :, :1], atol=0, rtol=0)
print('每头二范数保持、位置0不旋转：passed')

### 一整段与分块旋转必须一致；同内容的相对位置点积

先预测：同一个7位置输入，前3位置start=0、后4位置start=3是否等于整段start=0？
再固定两个内容向量，比较位置2/5与位置7/10，位置差都是3，点积应在容差内一致。
这不是说任意不同token只要距离相同就有相同分数。

In [ ]:
sequence = torch.randn(2, 2, 7, 4, generator=generator)
full = rope(sequence, start=0, theta=theta)
parts = torch.cat((rope(sequence[:, :, :3], start=0, theta=theta),
                   rope(sequence[:, :, 3:], start=3, theta=theta)), dim=2)
torch.testing.assert_close(parts, full, atol=1e-6, rtol=1e-6)
wrong = torch.cat((rope(sequence[:, :, :3], start=0, theta=theta),
                   rope(sequence[:, :, 3:], start=0, theta=theta)), dim=2)
assert not torch.allclose(wrong, full)
fixed_q = torch.tensor([[[[1., 2., 3., 4.]]]])
fixed_k = torch.tensor([[[[4., -1., 2., 3.]]]])
score_2_5 = (rope(fixed_q, 2, theta) * rope(fixed_k, 5, theta)).sum(-1)
score_7_10 = (rope(fixed_q, 7, theta) * rope(fixed_k, 10, theta)).sum(-1)
torch.testing.assert_close(score_2_5, score_7_10, atol=1e-5, rtol=1e-5)
print('整段/正确split最大误差:', (parts - full).abs().max().item())
print('错误重置start的最大误差:', (wrong - full).abs().max().item())
print('相同内容、相同位置差:', score_2_5.tolist(), score_7_10.tolist())

### RoPE负例与位置解释

head_dim必须是正偶数；offset=5只改变位置范围`[5,6,7]`，不增加维度。
RoPE不产生causal mask，未来是否可见仍由mask决定。

In [ ]:
bad = torch.randn(1, 1, 2, 3, generator=generator)
try:
    rope(bad)
except ValueError as error:
    print('expected odd-D rejection:', error)
else:
    raise AssertionError('Odd head dimension must be rejected.')
try:
    rope(torch.randn(1, 1, 2, 4, generator=generator), start=-1)
except ValueError as error:
    print('expected negative-offset rejection:', error)
else:
    raise AssertionError('Negative offset must be rejected.')

## 自测与参考答案

**1. RMSNorm会减均值吗？** 不会；只按最后特征维度计算平方均值并缩放。

**2. offset从0改5为什么shape不变？** B/H/T/D没变，只是使用了不同位置的旋转角度。

**3. RoPE负责屏蔽未来吗？** 不负责；RoPE注入位置，causal mask限制可见性。

**4. 为什么只旋转Q/K？** 本课程实现让位置影响匹配分数，V仍作为被汇集内容；以目标模型源码为准。

**5. 通过随机样例就够了吗？** 不够，要有边界输入、奇数D/负offset负例、不同offset、dtype和独立oracle。

**6. split-halves的4维怎么配对？** (0,2)、(1,3)。与相邻配对不是同一数组排列，不能无声混用。

**7. 为什么FP64输入也不能要求Mini RMSNorm达到1e-12？** 内部明确转FP32，.double()不改变这条计算路径。

**8. 今天完成独立实现了吗？** 本Notebook只提供已验证参考；学习者应另建自己的实现并记录预测、数值对齐与失败用例。

个人结果记入 [Day 8学习记录](../docs/DAY08_STUDY_LOG.md)，不要用参考输出自动勾选。